# 05 — Table 2 keyword extraction

**Public reproducibility version.** This notebook preserves the final analytical logic used for the revised manuscript. Local absolute paths and development-only cells were cleaned for portability. The notebook has not been re-executed as part of repository packaging.


# 05 — Table 2 keyword extraction for the five largest semantic clusters

Purpose:

- Use the **full set of texts assigned to each of the five largest semantic clusters**
- Extract data-driven keyword candidates using:
  - unigram + bigram frequency
  - document frequency
  - mean TF-IDF within each topic
  - topic-specific lift relative to the other four topics
- Produce a ranked candidate table for manual selection of the final 3–5 Table 2 keywords

This notebook does **not** rename topics and does **not** rerun SBERT or Leiden clustering.

Final topic labels used here:

1. Caregiving and Family Support  
2. Dementia Suspicions and Uncertainty  
3. Family Loss and Grief  
4. Alzheimer's Research on Diagnosis and Treatment  
5. Dementia Risk and Public Health

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

# ============================================================
# Repository-relative input/output paths
# ============================================================
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
MAIN_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "main_pipeline_no_brain_fog_final_v1"

POSTS_PATH = MAIN_OUTPUT_DIR / "filtered_posts_df_2_corrected_propagated.parquet"
TOPIC_MAP_PATH = MAIN_OUTPUT_DIR / "semantic_node_topic_map_revision.csv"

OUT_DIR = MAIN_OUTPUT_DIR / "table2_keyword_audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOPIC_LABELS = {
    0: "Caregiving and Family Support",
    1: "Suspicions and Uncertainty",
    2: "Family Loss and Grief",
    3: "Research on Diagnosis and Treatment",
    4: "Risk and Public Health",
}

print("MAIN_OUTPUT_DIR:", MAIN_OUTPUT_DIR)
print("OUT_DIR        :", OUT_DIR)


In [ ]:
# ============================================================
# 1. Validate and load inputs
# ============================================================
for p in [POSTS_PATH, TOPIC_MAP_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")

posts = pd.read_parquet(POSTS_PATH)
topic_map = pd.read_csv(TOPIC_MAP_PATH)

print("posts shape    :", posts.shape)
print("topic_map shape:", topic_map.shape)

required_posts = {"original_index", "text_processed"}
required_map = {"semantic_node_id", "original_index", "topic"}

missing_posts = required_posts - set(posts.columns)
missing_map = required_map - set(topic_map.columns)

if missing_posts:
    raise ValueError(f"Posts file missing columns: {sorted(missing_posts)}")
if missing_map:
    raise ValueError(f"Topic map missing columns: {sorted(missing_map)}")

In [ ]:
# ============================================================
# 2. Build one unique processed text per semantic node
# ============================================================

# A semantic node represents a unique processed text.
# We use original_index to recover the corresponding processed text
# from the propagated dataframe.

text_lookup = (
    posts[["original_index", "text_processed"]]
    .drop_duplicates(subset=["original_index"])
    .copy()
)

semantic = topic_map.merge(
    text_lookup,
    on="original_index",
    how="left",
    validate="one_to_one"
)

semantic = semantic.loc[
    semantic["topic"].isin(TOPIC_LABELS.keys())
].copy()

semantic["text_processed"] = (
    semantic["text_processed"]
    .fillna("")
    .astype(str)
    .str.strip()
)

semantic = semantic.loc[
    semantic["text_processed"].ne("")
].copy()

semantic["theme"] = semantic["topic"].map(TOPIC_LABELS)

topic_sizes = (
    semantic.groupby(["topic", "theme"])
    .size()
    .rename("n_texts")
    .reset_index()
)

print(topic_sizes.to_string(index=False))
print()
print("Total Top-5 semantic texts:", f"{len(semantic):,}")

topic_sizes.to_csv(
    OUT_DIR / "table2_top5_topic_sizes.csv",
    index=False
)

## Keyword extraction rules

We use:

- English stopword removal
- unigrams and bigrams
- minimum document frequency = 2
- exact generic ADRD labels (`dementia`, `alzheimer`, `alzheimers`, `adrd`) are excluded from the ranking because they are common across multiple clusters and contribute little thematic discrimination
- terms must appear in at least 3 texts within a topic to be eligible for the final ranking

The final ranking favors terms that are both:

1. important within the topic (mean TF-IDF), and
2. more characteristic of that topic than of the other four topics (topic lift)

In [ ]:
# ============================================================
# 3. Vectorize all Top-5 semantic texts
# ============================================================

texts = semantic["text_processed"].tolist()

vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
)

X_count = vectorizer.fit_transform(texts)
terms = np.array(vectorizer.get_feature_names_out())

# Remove only exact generic ADRD labels from candidate ranking.
GENERIC_EXACT_TERMS = {
    "dementia",
    "alzheimer",
    "alzheimers",
    "adrd",
}

keep_mask = np.array([
    term not in GENERIC_EXACT_TERMS
    for term in terms
])

X_count = X_count[:, keep_mask]
terms = terms[keep_mask]

tfidf_transformer = TfidfTransformer(
    norm="l2",
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=False,
)

X_tfidf = tfidf_transformer.fit_transform(X_count)

print("Texts used      :", f"{X_count.shape[0]:,}")
print("Candidate terms :", f"{X_count.shape[1]:,}")

In [ ]:
# ============================================================
# 4. Rank candidate terms within each topic
# ============================================================

topic_array = semantic["topic"].to_numpy()

MIN_TOPIC_DOC_FREQ = 3
EPS = 1e-6
MAX_LIFT_FOR_SCORE = 50.0

results = []

for topic in sorted(TOPIC_LABELS):
    in_topic = topic_array == topic
    out_topic = ~in_topic

    n_in = int(in_topic.sum())
    n_out = int(out_topic.sum())

    # Raw occurrence count.
    count_in = np.asarray(
        X_count[in_topic].sum(axis=0)
    ).ravel()

    # Number of topic texts containing each term.
    df_in = np.asarray(
        (X_count[in_topic] > 0).sum(axis=0)
    ).ravel()

    df_out = np.asarray(
        (X_count[out_topic] > 0).sum(axis=0)
    ).ravel()

    prevalence_in = df_in / max(n_in, 1)
    prevalence_out = df_out / max(n_out, 1)

    mean_tfidf_in = np.asarray(
        X_tfidf[in_topic].mean(axis=0)
    ).ravel()

    mean_tfidf_out = np.asarray(
        X_tfidf[out_topic].mean(axis=0)
    ).ravel()

    # Topic-specific lift based on mean TF-IDF.
    tfidf_lift = (
        (mean_tfidf_in + EPS)
        / (mean_tfidf_out + EPS)
    )

    capped_lift = np.minimum(tfidf_lift, MAX_LIFT_FOR_SCORE)

    # A transparent combined ranking:
    # within-topic importance × topic distinctiveness × recurrence.
    combined_score = (
        mean_tfidf_in
        * np.log1p(capped_lift)
        * np.sqrt(prevalence_in + EPS)
    )

    tmp = pd.DataFrame({
        "topic": topic,
        "display_topic": topic + 1,
        "theme": TOPIC_LABELS[topic],
        "term": terms,
        "count_in_topic": count_in,
        "n_texts_containing_term": df_in,
        "topic_text_prevalence": prevalence_in,
        "other_topic_text_prevalence": prevalence_out,
        "mean_tfidf_topic": mean_tfidf_in,
        "mean_tfidf_other_topics": mean_tfidf_out,
        "topic_lift": tfidf_lift,
        "combined_score": combined_score,
    })

    tmp = tmp.loc[
        tmp["n_texts_containing_term"] >= MIN_TOPIC_DOC_FREQ
    ].copy()

    tmp = tmp.sort_values(
        ["combined_score", "mean_tfidf_topic", "count_in_topic"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    tmp["rank"] = np.arange(1, len(tmp) + 1)

    results.append(tmp)

keyword_candidates = pd.concat(
    results,
    ignore_index=True
)

keyword_candidates.to_csv(
    OUT_DIR / "table2_keyword_candidates_ALL.csv",
    index=False
)

top30 = (
    keyword_candidates
    .loc[keyword_candidates["rank"] <= 30]
    .copy()
)

top30.to_csv(
    OUT_DIR / "table2_keyword_candidates_TOP30.csv",
    index=False
)

print("Saved full candidate table and Top-30 table.")

In [ ]:
# ============================================================
# 5. Also save pure frequency candidates for comparison
# ============================================================

freq_tables = []

for topic in sorted(TOPIC_LABELS):
    tmp = keyword_candidates.loc[
        keyword_candidates["topic"] == topic
    ].copy()

    tmp = tmp.sort_values(
        ["n_texts_containing_term", "count_in_topic"],
        ascending=[False, False]
    ).head(30)

    tmp["frequency_rank"] = np.arange(1, len(tmp) + 1)

    freq_tables.append(tmp)

top30_frequency = pd.concat(
    freq_tables,
    ignore_index=True
)

top30_frequency.to_csv(
    OUT_DIR / "table2_keyword_candidates_TOP30_BY_FREQUENCY.csv",
    index=False
)

print("Saved Top-30 frequency comparison.")

In [ ]:
# ============================================================
# 6. Print the Top 20 candidates for each topic
# ============================================================

display_cols = [
    "rank",
    "term",
    "count_in_topic",
    "n_texts_containing_term",
    "topic_text_prevalence",
    "mean_tfidf_topic",
    "topic_lift",
    "combined_score",
]

for topic in sorted(TOPIC_LABELS):
    print("\n" + "=" * 100)
    print(f"TOPIC {topic + 1}: {TOPIC_LABELS[topic]}")
    print("=" * 100)

    print(
        top30.loc[
            top30["topic"] == topic,
            display_cols
        ]
        .head(20)
        .to_string(index=False)
    )

## Manual review step

Do not mechanically use ranks 1–5 as the final Table 2 keywords.

Review the Top-30 candidate list and select 3–5 terms per topic that satisfy all three criteria:

1. reasonably frequent within the topic,
2. relatively topic-distinctive,
3. collectively cover different aspects of the topic rather than repeating near-synonyms.

The next cell creates a review sheet for this manual selection.

In [ ]:
# ============================================================
# 7. Create a review sheet for final Table 2 selection
# ============================================================

review = top30[[
    "display_topic",
    "theme",
    "rank",
    "term",
    "count_in_topic",
    "n_texts_containing_term",
    "topic_text_prevalence",
    "mean_tfidf_topic",
    "topic_lift",
    "combined_score",
]].copy()

review["Selected_for_Table2"] = ""
review["Reviewer_note"] = ""

review.to_csv(
    OUT_DIR / "table2_keyword_manual_review_sheet.csv",
    index=False
)

print("Manual review sheet saved:")
print(OUT_DIR / "table2_keyword_manual_review_sheet.csv")

In [ ]:
# ============================================================
# 8. Optional placeholder for final frozen Table 2 keywords
#    Fill this ONLY AFTER reviewing the candidate tables.
# ============================================================

FINAL_KEYWORDS = {
    0: [],
    1: [],
    2: [],
    3: [],
    4: [],
}

final_rows = []

for topic in sorted(TOPIC_LABELS):
    final_rows.append({
        "Topic": topic + 1,
        "Theme": TOPIC_LABELS[topic],
        "Final Sample Keywords": ", ".join(FINAL_KEYWORDS[topic]),
    })

final_keyword_table = pd.DataFrame(final_rows)

final_keyword_table.to_csv(
    OUT_DIR / "table2_final_keywords_TEMPLATE.csv",
    index=False
)

print(final_keyword_table.to_string(index=False))
print()
print("Template saved. Fill FINAL_KEYWORDS only after reviewing candidates.")

## Output files

All outputs are written to:

`main_pipeline_no_brain_fog_final_v1/table2_keyword_audit/`

Key files:

- `table2_top5_topic_sizes.csv`
- `table2_keyword_candidates_ALL.csv`
- `table2_keyword_candidates_TOP30.csv`
- `table2_keyword_candidates_TOP30_BY_FREQUENCY.csv`
- `table2_keyword_manual_review_sheet.csv`
- `table2_final_keywords_TEMPLATE.csv`

For manuscript documentation, the most defensible description is:

> Sample keywords were selected from high-frequency and topic-distinctive terms identified using TF-IDF-based ranking within each semantic cluster, followed by manual review for interpretability and thematic coverage.